In [22]:
import pandas as pd
import torch
import numpy as np
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report
)
import os

In [23]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [24]:
train_df = pd.read_csv("/kaggle/input/datasets/zahrapriyono/emotion-classification-in-e-commerce/Emotion-Classification-in-E-Commerce/data/processed/train.csv")
val_df = pd.read_csv("/kaggle/input/datasets/zahrapriyono/emotion-classification-in-e-commerce/Emotion-Classification-in-E-Commerce/data/processed/val.csv")
test_df = pd.read_csv("/kaggle/input/datasets/zahrapriyono/emotion-classification-in-e-commerce/Emotion-Classification-in-E-Commerce/data/processed/test.csv")
emotion_names = ['Happy', 'Love', 'Sadness', 'Fear', 'Anger']

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"Label distribution (train):\n{train_df['label'].value_counts().sort_index()}")

Train: 24404 | Val: 3051 | Test: 3051
Label distribution (train):
label
0    14911
1     6847
2     1653
3      320
4      673
Name: count, dtype: int64


In [25]:
# === IMPROVEMENT #1: Class weights ===
from sklearn.utils.class_weight import compute_class_weight

labels_array = train_df['label'].values
weights = compute_class_weight('balanced', classes=np.unique(labels_array), y=labels_array)
class_weights = torch.tensor(weights, dtype=torch.float).to(device)
print("Class weights:", dict(zip(np.unique(labels_array), weights.round(3))))

Class weights: {np.int64(0): np.float64(0.327), np.int64(1): np.float64(0.713), np.int64(2): np.float64(2.953), np.int64(3): np.float64(15.252), np.int64(4): np.float64(7.252)}


In [26]:
MODEL_NAME = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Quick sanity check
sample = tokenizer("Good product, fast delivery!", truncation=True, max_length=128)
print("Sample token ids:", sample["input_ids"][:10], "...")

Sample token ids: [0, 18621, 12996, 4, 4271, 117989, 38, 2] ...


In [27]:
class EmotionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        if isinstance(idx, list):
            texts  = [str(self.texts[i]) for i in idx]
            labels = [self.labels[i] for i in idx]
            encoding = self.tokenizer(
                texts, truncation=True,
                padding='max_length', max_length=self.max_length,
                return_tensors='pt'
            )
            encoding['labels'] = torch.tensor(labels, dtype=torch.long)
            return encoding

        encoding = self.tokenizer(
            str(self.texts[idx]), truncation=True,
            padding='max_length', max_length=self.max_length,
            return_tensors='pt'
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

In [28]:
train_dataset = EmotionDataset(
    train_df['text'].tolist(), train_df['label'].tolist(), tokenizer
)
val_dataset = EmotionDataset(
    val_df['text'].tolist(), val_df['label'].tolist(), tokenizer
)
test_dataset = EmotionDataset(
    test_df['text'].tolist(), test_df['label'].tolist(), tokenizer
)

print(f"Sample item keys: {list(train_dataset[0].keys())}")
print(f"input_ids shape: {train_dataset[0]['input_ids'].shape}")

Sample item keys: ['input_ids', 'attention_mask', 'labels']
input_ids shape: torch.Size([128])


In [29]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=5
)
model.to(device)
print(f"Model loaded: {MODEL_NAME}")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded: xlm-roberta-base
Trainable params: 278,047,493


In [30]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds  = pred.predictions.argmax(-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='weighted', zero_division=0
    )
    acc = accuracy_score(labels, preds)

    emotion_names = ['Happy', 'Love', 'Sadness', 'Fear', 'Anger']
    _, _, per_class_f1, _ = precision_recall_fscore_support(
        labels, preds, average=None,
        labels=list(range(5)), zero_division=0
    )
    per_class = {f"f1_{emotion_names[i]}": round(float(per_class_f1[i]), 4)
                 for i in range(5)}

    return {"accuracy": acc, "f1": f1,
            "precision": precision, "recall": recall, **per_class}

In [31]:
# === IMPROVEMENT #2: WeightedTrainer ===
from transformers import Trainer
import torch.nn as nn

class WeightedTrainer(Trainer):
    def __init__(self, class_weights, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fn = nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

In [32]:
os.makedirs("../models/xlmr", exist_ok=True)
os.makedirs("../results", exist_ok=True)

training_args = TrainingArguments(
    output_dir="../models/xlmr",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,           # ← tambah ini
    fp16=torch.cuda.is_available(),
    report_to="none",
    logging_steps=100,
    remove_unused_columns=False        # ← tambah ini
)

print(training_args)

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=Fals

In [33]:
trainer = WeightedTrainer(
    class_weights = class_weights,
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [34]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,F1 Happy,F1 Love,F1 Sadness,F1 Fear,F1 Anger
1,1.141258,1.086315,0.585710,0.596464,0.656603,0.585710,0.645600,0.559100,0.467600,0.084500,0.446900
2,1.041616,1.185953,0.533596,0.528323,0.671528,0.533596,0.528200,0.553600,0.523900,0.156900,0.461500
3,0.985323,1.174787,0.605048,0.616855,0.669758,0.605048,0.670600,0.573600,0.439100,0.204100,0.495900
4,0.835444,1.194944,0.633235,0.642884,0.669597,0.633235,0.707500,0.567100,0.517400,0.202200,0.497200
5,0.760574,1.262782,0.633891,0.643690,0.668847,0.633891,0.712200,0.569500,0.492400,0.164700,0.478300


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=3815, training_loss=0.9581403274836109, metrics={'train_runtime': 2601.8981, 'train_samples_per_second': 46.897, 'train_steps_per_second': 1.466, 'total_flos': 8026418935864320.0, 'train_loss': 0.9581403274836109, 'epoch': 5.0})

In [35]:
eval_results = trainer.evaluate(test_dataset)
print("Test Set Results:", eval_results)

Test Set Results: {'eval_loss': 1.1583572626113892, 'eval_accuracy': 0.6361848574237955, 'eval_f1': 0.6447853490629616, 'eval_precision': 0.6649802539981694, 'eval_recall': 0.6361848574237955, 'eval_f1_Happy': 0.7178, 'eval_f1_Love': 0.5478, 'eval_f1_Sadness': 0.525, 'eval_f1_Fear': 0.2273, 'eval_f1_Anger': 0.5059, 'eval_runtime': 22.0467, 'eval_samples_per_second': 138.388, 'eval_steps_per_second': 4.354, 'epoch': 5.0}


In [36]:
output = trainer.predict(test_dataset)
logits = torch.from_numpy(output.predictions)
probs  = torch.nn.functional.softmax(logits, dim=-1)
preds  = torch.argmax(probs, dim=-1).numpy()

print("\nClassification Report:")
print(classification_report(
    output.label_ids, preds,
    target_names=emotion_names,
    zero_division=0
))


Classification Report:
              precision    recall  f1-score   support

       Happy       0.78      0.66      0.72      1864
        Love       0.51      0.59      0.55       856
     Sadness       0.43      0.69      0.52       207
        Fear       0.21      0.25      0.23        40
       Anger       0.50      0.51      0.51        84

    accuracy                           0.64      3051
   macro avg       0.48      0.54      0.50      3051
weighted avg       0.66      0.64      0.64      3051



In [37]:
# Save model
trainer.save_model("../models/xlmr/xlmr_emotion_model")

# Save predictions CSV
df_results = pd.DataFrame({
    'text':          test_df['text'].values,
    'true_label_id': output.label_ids,
    'pred_label_id': preds,
    'true_emotion':  [emotion_names[i] for i in output.label_ids],
    'pred_emotion':  [emotion_names[i] for i in preds]
})
df_results.to_csv("../results/xlmr_results.csv", index=False)
print("Saved to ../results/xlmr_results.csv")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to ../results/xlmr_results.csv


In [1]:
import pandas as pd, os

os.makedirs("/kaggle/working/results", exist_ok=True)

xlmr_results = {
    'model': 'XLM-R',
    'accuracy': 0.64,
    'weighted_f1': 0.64,
    'macro_f1': 0.50,
    'f1_Happy': 0.72,
    'f1_Love': 0.55,
    'f1_Sadness': 0.52,
    'f1_Fear': 0.23,
    'f1_Anger': 0.51
}

pd.DataFrame([xlmr_results]).to_csv("/kaggle/working/results/xlmr_results.csv", index=False)
print("Saved: xlmr_results.csv")

Saved: xlmr_results.csv
